In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import torch
import json
import re
import gc
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

class PhiModelService:
    """Handles model loading, quantization, and raw text generation."""
    def __init__(self, model_id="microsoft/Phi-3.5-mini-instruct"):
        self.model_id = model_id
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = None
        self.pipeline = None
        self.model = None
        self._initialize_resources()

    def _initialize_resources(self):
        print(f"[Service] Initializing {self.model_id} on {self.device}...")

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            llm_int8_enable_fp32_cpu_offload=True
        )

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=quantization_config,
            trust_remote_code=False,
            device_map="auto"
        )

        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer
        )

    def generate(self, messages, max_tokens=500):
        """Generates text response from the loaded model pipeline."""
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        outputs = self.pipeline(
            prompt,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=True,
            return_full_text=False
        )
        return outputs[0]['generated_text']

# --- Initialization & OOM Cleanup ---
def setup_service(force=False):
    global model_service
    if force and 'model_service' in globals():
        print("🧹 Force resetting Model Service...")
        del globals()['model_service']
        gc.collect()
        torch.cuda.empty_cache()

    if 'model_service' not in globals():
        model_service = PhiModelService()
    else:
        print("✅ Re-using active Model Service.")
    return model_service

# Run initialization
model_service = setup_service(force=False)

In [ ]:
class PhiOrchestrator:
    """Handles orchestration logic, parsing, and validation with dedicated repair functions."""
    def __init__(self, model_service):
        self.service = model_service
        self.allowed_actions = ["Create Ticket", "Ask for more info", "Reject the request"]

    def extract_json(self, raw_text):
        try:
            match = re.search(r'\{.*\}', raw_text, re.DOTALL)
            if match:
                return json.loads(match.group().strip())
            return None
        except Exception:
            return None

    def validate_decision(self, data):
        if not data or "decision" not in data:
            return False, "Invalid format or missing 'decision' key."

        decision = data["decision"]
        if decision in self.allowed_actions:
            return True, f"Valid action: {decision}"
        return False, f"Invalid action '{decision}'. Must be one of {self.allowed_actions}"

    def repair_response(self, original_messages, faulty_output):
        """Isolated function to attempt a repair pass on invalid outputs."""
        print("⚠️ Attempting to repair invalid model response...")

        repair_messages = original_messages.copy()
        repair_messages.append({"role": "assistant", "content": faulty_output})
        repair_messages.append({
            "role": "user",
            "content": "Your previous response was not a valid JSON or the decision was incorrect. "
                       "Please provide the output again. Ensure it is ONLY the JSON block with a valid 'decision'."
        })

        new_raw_output = self.service.generate(repair_messages)
        new_extracted_obj = self.extract_json(new_raw_output)
        is_valid, validation_msg = self.validate_decision(new_extracted_obj)

        return {
            "data": new_extracted_obj,
            "is_valid": is_valid,
            "validation_message": f"[Repair Pass] {validation_msg}"
        }, new_raw_output

    def ask_llm_guarded(self, message):
        system_prompt = (
            "You are an orchestrator. You MUST respond ONLY with a JSON block. "
            f"The 'decision' field must be exactly one of: {self.allowed_actions}. "
            "Include 'role' and 'reason'."
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message},
        ]

        raw_output = self.service.generate(messages)
        extracted_obj = self.extract_json(raw_output)
        is_valid, validation_msg = self.validate_decision(extracted_obj)

        if not is_valid:
            return self.repair_response(messages, raw_output)

        return {
            "data": extracted_obj,
            "is_valid": is_valid,
            "validation_message": validation_msg
        }, raw_output

# --- OOM Recovery Logic ---
# If you keep getting OOM, uncomment the next two lines to force a complete reset:
# if 'model_service' in globals(): del model_service
# gc.collect(); torch.cuda.empty_cache()

if 'model_service' not in globals():
    print("🚀 Initializing Model Service...")
    model_service = PhiModelService()
else:
    print("✅ Re-using active Model Service.")

orchestrator = PhiOrchestrator(model_service)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

class ResultRenderer:
    """Handles the visual rendering of orchestration outcomes."""
    @staticmethod
    def render_success(data, validation_msg, raw_logs):
        print("=== ORCHESTRATION RESULT ===")
        print(f"✅ Status: {validation_msg}")

        if data:
            print("\nCLEAN RESPONSE:")
            for key, value in data.items():
                print(f"- {key.capitalize()}: {value}")

        ResultRenderer._render_logs(raw_logs)

    @staticmethod
    def render_failure(validation_msg, raw_logs, data=None):
        print("=== ORCHESTRATION RESULT ===")
        print(f"❌ Status: {validation_msg}")

        if data:
            print("\nPARTIAL DATA EXTRACTED:")
            print(json.dumps(data, indent=2))

        ResultRenderer._render_logs(raw_logs)

    @staticmethod
    def _render_logs(raw_logs):
        log_view = widgets.HTML(f'<pre style="white-space: pre-wrap; font-size: 12px;">{raw_logs}</pre>')
        accordion = widgets.Accordion(children=[log_view])
        accordion.set_title(0, "View Raw Model Logs")
        display(accordion)

class OrchestratorUI:
    """Handles layout management and user interactions."""
    def __init__(self, orchestrator_instance):
        self.orchestrator = orchestrator_instance

        self.input_text = widgets.Textarea(
            value='Initiate system diagnostic',
            placeholder='Enter orchestration command...',
            description='Task:',
            layout=widgets.Layout(width='90%', height='100px')
        )
        self.run_button = widgets.Button(
            description='Execute Orchestration',
            button_style='primary',
            icon='bolt',
            layout=widgets.Layout(width='90%')
        )
        self.output_area = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '15px', 'margin': '10px 0'})
        self.run_button.on_click(self.handle_execution)

    def handle_execution(self, b):
        with self.output_area:
            clear_output()
            print("⏳ Processing request...")

            result_context, raw_logs = self.orchestrator.ask_llm_guarded(self.input_text.value)

            clear_output()
            if result_context['is_valid']:
                ResultRenderer.render_success(result_context['data'], result_context['validation_message'], raw_logs)
            else:
                ResultRenderer.render_failure(result_context['validation_message'], raw_logs, result_context['data'])

    def render(self):
        display(widgets.HTML("<h3>Phi-3.5 Guarded Orchestrator</h3>"))
        display(self.input_text, self.run_button, self.output_area)

# Instantiate and render
ui = OrchestratorUI(orchestrator)
ui.render()